In [ ]:
import geopandas as gpd

from benchmark_pipeline.path import DATA
from benchmark_pipeline.generator.template_question.multihop.generator_multihop import make_multihop_questions
from benchmark_pipeline.generator.template_question.multihop.make_multihop_text import make_text_multihop

In [2]:
repo = "benchmark_1K_after_correctionbug"
df_question = gpd.read_parquet(DATA / repo / "tables" / "df_question.parquet")
df_street = gpd.read_parquet(DATA / repo / "tables" / "df_street.parquet")
df_area = gpd.read_parquet(DATA / repo / "tables" / "df_area.parquet")
df_osm = gpd.read_parquet(DATA / repo / "tables" / "df_osm.parquet")

In [3]:
%load_ext autoreload
%autoreload 2
import numpy as np

id_start = df_question[df_question["values"].map(tuple) == ("station",)].index[5]


backward_index, df_question, df_multihop = make_multihop_questions(id_start, {5:0, 4: 3, 3: 3, 2: 3, 1: 2}, df_question, df_osm, df_street, df_area)

In [4]:
df_multihop = None
k=1000
for id_start in df_question.sample(k, random_state=0).index:
    backward_index, df_question, df_multihop = make_multihop_questions(
        id_start, {5: 0, 4: 3, 3: 3, 2: 3, 1: 2}, df_question, df_osm, df_street, df_area, df_multihop=df_multihop)
df_multihop = make_text_multihop(df_multihop, df_question)


In [5]:
from benchmark_pipeline.config import MULTIHOP_COLUMNS


def dicts_to_lists(df):
    # Parquet only takes str dict keys: node i is stored at position i - 1, each dic_query value as a list
    def as_list(value):
        return value if isinstance(value, list) else [] if value is None else [value]
    return df.assign(
        template=df["template"].map(lambda d: [d[node] for node in sorted(d)] if isinstance(d, dict) else None),
        dic_query=df["dic_query"].map(lambda d: [as_list(d[node]) for node in sorted(d)] if isinstance(d, dict) else None))


multihop_dir = DATA / "multihop"
multihop_dir.mkdir(parents=True, exist_ok=True)
# Lateral rows keep template columns such as area_geometry, which parquet rejects
dicts_to_lists(df_question[MULTIHOP_COLUMNS]).to_parquet(multihop_dir / "df_question.parquet")
dicts_to_lists(df_multihop).to_parquet(multihop_dir / "df_multihop.parquet")

In [15]:
def sample_by_depth(df_multihop, k, seed=0):
    """k random questions per depth, all of them when a depth has fewer.

    Args:
        df_multihop (DataFrame): multihop questions, `template` as a dict or a list.
        k (int): questions kept per depth.
        seed (int): random seed.

    Returns:
        DataFrame: `query_texte` and `answers`, sorted by depth.
    """
    # template has one entry per node, plus a closing 0
    depth = df_multihop["template"].map(len) - 1
    shuffled_depth = depth.sample(frac=1, random_state=seed)
    kept_depth = shuffled_depth.groupby(shuffled_depth).head(k).sort_values(kind="stable")
    dump = df_multihop.loc[kept_depth.index, ["description_textuelle", "answers"]]
    dump = dump.rename(columns={"description_textuelle": "query"}).assign(depth=kept_depth)
    return dump